# Stage 12 — original 72-case Track 2a GPU inference, with durable recovery

**PolicyProof AI · Lexsi Hackathon · experimental owner workflow.**

This notebook makes the prepared, guarded `scripts.challenge72_gpu_inference` executable in Google Colab. It does not run by itself and does **not** claim a completed experiment, a trained model or semantic accuracy.

**Why this matters:** new Track 2a contains 72 frozen multilingual adversarial cases with no original model outputs. On an authorized T4 runtime, the runner saves **each original raw response immediately**. When you explicitly mount your private Google Drive and clone the repository there, these source-bound checkpoint files survive a disconnected Colab VM. Re-running validates the existing prefix and generates only missing cases.

**Follow these rules:** obtain legitimate gated access to `CohereLabs/tiny-aya-global`, read its noncommercial/license terms, and personally authorize model inference. Do not upload the final-60 holdout or private human reviews. Do not share HF tokens, cached weights, or reviewer notes. This notebook uses the previously requested base-model revision `af89d219b53ed9b13b8a4645f9c8028973510324`; this is a **recorded request**, not independent proof of loaded weights. The real runner checks the observed model SHA on GPU.

**Default:** `RUN_ORIGINAL_72_GPU=False`, `RUN_CPU_AUDITKIT=False`, `EXPORT_RESULTS=False`. Nothing is trained or evaluated merely by executing the setup cells. The CPU-only preflight can be rerun safely.

**Estimated research status before owner execution:** 48 saved development baseline cases; 72 newly curated challenges **unscored**. Colab quota and model access are not guaranteed.

## 1 — Configure private storage and all action switches

Use **Runtime → Change runtime type → T4 GPU**. Private Google Drive storage is recommended for checkpoints; mounting it requires an explicit Google approval dialog. If you do not want Drive, set `SAVE_CHECKPOINTS_IN_PRIVATE_DRIVE=False`; files will remain on ephemeral Colab storage and can be lost if the VM disconnects.

Do **not** turn on the GPU switch until the frozen-data preflight and Hugging Face access checks pass.

In [ ]:
from pathlib import Path
import re
SAVE_CHECKPOINTS_IN_PRIVATE_DRIVE = True   # private user-approved Drive, not GitHub
VERIFY_HF_ACCOUNT_AND_MODEL_SHA = False   # change to True to check your gated model access
MODEL_LICENSE_AND_COST_APPROVED = False    # only YOU may change after reviewing terms
RUN_ORIGINAL_72_GPU = False               # OFF; change to True to run genuine original-model CUDA inference
RUN_CPU_AUDITKIT = False                 # OFF; True only after all 72 original responses were saved
RUN_GUARD_COMPARISON = False               # OFF; CPU-only analysis only after original AuditKIT replay
EXPORT_RESULTS = False                   # OFF; True only after CPU replay succeeds
EXPORT_PARTIAL_BACKUP = False            # OFF; optional preserved checkpoint snapshot for manual download

# Previously requested, fully spelled-out revision from your original 48-case development evaluation.
# This SHA is NOT treated as independently observed GPU weights.
MODEL_SHA = "af89d219b53ed9b13b8a4645f9c8028973510324"
assert re.fullmatch(r"[0-9a-f]{40}", MODEL_SHA)
REPOSITORY = "https://github.com/betterlifestyle04-commits/PolicyProof-AI.git"
FROZEN_DATA = "artifacts/track2a_challenge_unscored/challenge_v1.jsonl"
RAW_REPORT = "artifacts/track2a_72_unchanged_base_predictions.json"
CHECKPOINT = "artifacts/challenge72_gpu_progress"
AUDIT_DIR = "artifacts/actual_72_case_evaluation"
print("Selections — Drive:", SAVE_CHECKPOINTS_IN_PRIVATE_DRIVE,
      "| access check:", VERIFY_HF_ACCOUNT_AND_MODEL_SHA,
      "| GPU:", RUN_ORIGINAL_72_GPU,
      "| AuditKIT:", RUN_CPU_AUDITKIT, "| download:", EXPORT_RESULTS)
print("Training: NEVER executed by this notebook.")

## 2 — Open the exact project source on PRIVATE persistent storage

For Drive users, the project itself is cloned under `MyDrive/PolicyProof_Lexsi_72/PolicyProof-AI`. It is **not** shared publicly by this notebook. The `artifacts/` checkpoint is inside that same folder: each case file is written once in the existing Drive directory.

**Restart safety:** on reconnect this cell reuses the original checkout **without pulling a newer commit**. This prevents a new branch or changed generator from silently changing evidence. If you intentionally want newer code, use a NEW project folder, a fresh checkpoint and a new experiment; never overwrite prior data.

In [ ]:
import os, subprocess, sys
import torch
assert torch.cuda.is_available(), "No CUDA runtime. Select T4 GPU first."
print("Connected GPU:", torch.cuda.get_device_name(0))
if SAVE_CHECKPOINTS_IN_PRIVATE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    private_root = Path("/content/drive/MyDrive/PolicyProof_Lexsi_72")
else:
    private_root = Path("/content/PolicyProof_Lexsi_72_EPHEMERAL")
private_root.mkdir(parents=True, exist_ok=True)
ROOT = private_root / "PolicyProof-AI"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)
else:
    assert (ROOT / ".git").is_dir(), "Existing path is not an unchanged Git checkout."
    remote = subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    assert remote in (REPOSITORY, REPOSITORY.removesuffix(".git")), (
        "Refusing unrelated or substituted repository on persistent storage."
    )
code_sha = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip()
assert re.fullmatch(r"[0-9a-f]{40}", code_sha)
assert (ROOT / "scripts/challenge72_gpu_inference.py").is_file()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Original preserved project source commit:", code_sha)
print("Private work directory:", ROOT)
print("No GPU model loaded; no training performed.")
if not SAVE_CHECKPOINTS_IN_PRIVATE_DRIVE:
    print("WARNING: /content is EPHEMERAL. Manually download every original checkpoint.")

## 3 — Install versioned inference and CPU-AuditKIT libraries

This modifies only the connected Colab Python environment. It does not download the Tiny Aya weights or call the GPU model. Restart the runtime if `pip` indicates an incompatible already-imported package, then rerun setup. The original GPU runner checks actual loaded-model revision before saving any response.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--prefer-binary",
    "bitsandbytes==0.50.2", "auditkit==1.1.3",
    "transformers>=4.56,<5", "accelerate>=1.3,<2",
    "huggingface_hub>=0.35,<2",
], check=True)
import importlib.metadata as im
for name in ("bitsandbytes", "auditkit", "transformers", "accelerate", "huggingface_hub"):
    print(name, im.version(name))
assert im.version("auditkit") == "1.1.3", "Require pinned native AuditKIT 1.1.3"
print("Research dependencies installed. No model inference or training occurred.")

## 4 — Freeze the EXACT 72 challenge cases, then run the safe CPU preflight

The original challenge comes **only** from the pinned source code. If it already exists, it is reused and its SHA is checked; it is **never overwritten**. The expected SHA-256 is `cdd5cf4e54caa6c79ce1f0cd5bd0f05344f0dc22039a2ad56a3c1c0ee2d32cd1`.

A previous checkpoint automatically sets `--resume` during preflight. This validates each existing case record, source hash, ID order and loaded model revision, without calling a model.

In [ ]:
import json
from scripts.score_track2a_72 import KNOWN_CHALLENGE_SHA256
from scripts.challenge72_gpu_inference import preflight
from policyproof.core import read_jsonl, sha256
source_file = ROOT / FROZEN_DATA
if not source_file.is_file():
    subprocess.run([
        sys.executable, "-m", "scripts.generate_track2a_challenge",
        "--output-dir", str(source_file.parent),
    ], cwd=ROOT, check=True)
assert sha256(source_file) == KNOWN_CHALLENGE_SHA256, (
    "Frozen challenge was changed. STOP; never modify gold labels."
)
cases = read_jsonl(source_file)
assert len(cases) == 72
original_report = ROOT / RAW_REPORT
progress_dir = ROOT / CHECKPOINT
RESUME_ORIGINAL_CHECKPOINT = progress_dir.exists()
if original_report.exists():
    print("Original full report already exists. NEVER rerun inference or overwrite it.")
    print("Proceed to the CPU-only score/verification cell.")
else:
    pre = preflight(ROOT, dataset=FROZEN_DATA, revision=MODEL_SHA,
                    output=RAW_REPORT, checkpoint_dir=CHECKPOINT,
                    resume=RESUME_ORIGINAL_CHECKPOINT)
    print(json.dumps(pre, indent=2, ensure_ascii=False))
    assert pre["status"] == "READY_FOR_EXPLICIT_GPU_EXECUTION", (
        "Stop: frozen source, original model SHA or checkpoint failed read-only preflight."
    )
print("Original challenge SHA:", KNOWN_CHALLENGE_SHA256)
print("Checkpoint present:", RESUME_ORIGINAL_CHECKPOINT)
print("GPU model has NOT been invoked.")

## 5 — Optional gated Hugging Face access and source revision verification

**Only set `VERIFY_HF_ACCOUNT_AND_MODEL_SHA=True` when you are ready.** This accesses the public/gated Hugging Face model metadata using your own account and may require an interactive secure login. **Never paste a token into the notebook code or this chat.** This checks only that the requested commit exists and your account can query it. The actual model loader separately checks the SHA exposed by loaded weights during a genuine GPU run.

The model's license and noncommercial restrictions are your responsibility to review and approve before inference.

In [ ]:
REMOTE_SHA_VERIFIED = False
if VERIFY_HF_ACCOUNT_AND_MODEL_SHA:
    from huggingface_hub import HfApi, notebook_login, whoami
    try:
        account = whoami()
        print("Hugging Face account:", account.get("name", "authorized account"))
        details = HfApi().model_info("CohereLabs/tiny-aya-global", revision=MODEL_SHA)
    except Exception:
        print("Login through the SECURE Hugging Face widget, not notebook source.")
        notebook_login()
        details = HfApi().model_info("CohereLabs/tiny-aya-global", revision=MODEL_SHA)
    assert details.sha == MODEL_SHA, "Hub returned a different revision: stop."
    REMOTE_SHA_VERIFIED = True
    print("Authorized Hub revision metadata checked:", details.sha)
else:
    print("Model access NOT verified. CPU preflight still does not require authentication.")
print("No original GPU inference executed by this cell.")

## 6 — EXPLICIT owner opt-in to original 72-case CUDA inference

**STOP unless** the CPU preflight passed, you have legitimate gated Tiny Aya model access, have read and accepted the applicable license/compute costs, and the original report does not already exist.

To start, return to Cell 1 and set **all three** values `VERIFY_HF_ACCOUNT_AND_MODEL_SHA=True`, `MODEL_LICENSE_AND_COST_APPROVED=True` and `RUN_ORIGINAL_72_GPU=True`. Then rerun the configuration, access, preflight and this cell.

The unchanged original base model loads once; each successful response is written as an immutable source-bound `case_000.json ... case_071.json` under the Drive-backed checkpoint. A disconnected VM leaves all successfully written checkpoint files in your private Drive directory. On reconnect, rerun cells in order; `RESUME_ORIGINAL_CHECKPOINT` will be true, and the runner regenerates **only unfinished cases**. No fine-tuning takes place.

If the requested full 40-hex revision differs from the loaded model's actual observed revision, the runner fails closed. The generated report still records **self-reported** accelerator metadata, not independent third-party GPU attestation.

In [ ]:
if RUN_ORIGINAL_72_GPU:
    assert VERIFY_HF_ACCOUNT_AND_MODEL_SHA is True, "Explicit HF account verification required"
    assert REMOTE_SHA_VERIFIED is True, "Actual gated model SHA/access check missing"
    assert MODEL_LICENSE_AND_COST_APPROVED is True, (
        "You must personally approve model license and compute costs"
    )
    assert not original_report.exists(), (
        "Original 72 model report already exists. Never overwrite; score it instead."
    )
    from scripts.challenge72_gpu_inference import execute_gpu
    result = execute_gpu(
        ROOT, FROZEN_DATA, MODEL_SHA, RAW_REPORT,
        checkpoint_dir=CHECKPOINT, resume=RESUME_ORIGINAL_CHECKPOINT,
    )
    print(json.dumps(result, indent=2, ensure_ascii=False))
    assert (ROOT / RAW_REPORT).is_file()
    print("Original raw responses saved in PRIVATE Drive/repository artifacts.")
    print("This is original-model inference, NOT training or independent semantic assessment.")
else:
    print("GPU execution remains OFF. No Tiny Aya model loaded or generated outputs created.")

## 7 — Optional native AuditKIT 1.1.3 CPU scoring and full source replay

Set `RUN_CPU_AUDITKIT=True` in Cell 1 **only after the actual original 72-output report exists**. The script checks all 72 raw IDs, exact frozen dataset SHA, model identity/revision metadata, deterministic generation settings, and genuine native AuditKIT replay. It writes a new three-file evaluation receipt once, and refuses overwrites.

**CRITICAL:** these are deterministic synthetic-label diagnostics, not human-judged factual accuracy. A completion marker is not enough; the independent replay must agree. A fabricated self-reported GPU metadata field may still be internally consistent; preserve unedited Colab runtime logs.

In [ ]:
if RUN_CPU_AUDITKIT:
    assert original_report.is_file(), "No genuine original model output report exists"
    from scripts.score_track2a_72 import score_saved_72
    from scripts.verify_track2a_72_receipts import verify_completed_72
    target_dir = ROOT / AUDIT_DIR
    if not target_dir.exists():
        measured, native = score_saved_72(
            source_file, original_report, AUDIT_DIR, root=ROOT
        )
        print("Saved actual CPU AuditKIT measured 72-case report and native receipt.")
    verified = verify_completed_72(AUDIT_DIR, FROZEN_DATA, RAW_REPORT, root=ROOT)
    assert verified["status"] == "VERIFIED_LOCAL_72_CASE_SOURCE_REPLAY"
    assert verified["original_gpu_provenance_independently_authenticated"] is False
    print(json.dumps(verified, indent=2, ensure_ascii=False))
else:
    print("Native AuditKIT scoring OFF; no evaluation receipts generated.")

## 7b — Optional CPU-only paired output-guard analysis (NO GPU / NO NEW MODEL RUN)

This optional study must follow the **original 72 saved GPU outputs** and the separately verified **native AuditKIT CPU replay**. Set `RUN_GUARD_COMPARISON=True` in Cell 1 only after those exist. It does not run Tiny Aya or any optimizer, modify the original outputs, retrain weights or use reference answers when applying the guard.

The result is one write-once aggregate-only JSON report showing schema, exact-reference, citation, unsupported abstention and response-coverage changes. Each attack family is compared against its 12 **matched** control tasks (not 72 independent real-world policies). Output only describes a synthetic research diagnostic; human semantic evaluation is not established.


In [ ]:
if RUN_GUARD_COMPARISON:
    assert original_report.is_file(), 'No original 72 raw GPU output report found'
    from scripts.verify_track2a_72_receipts import verify_completed_72
    checked_source = verify_completed_72(AUDIT_DIR, FROZEN_DATA, RAW_REPORT, root=ROOT)
    assert checked_source['status'] == 'VERIFIED_LOCAL_72_CASE_SOURCE_REPLAY'
    from scripts.challenge72_guard_effect import audit_from_saved_files
    paired_guard = audit_from_saved_files(
        source_file, original_report, root=ROOT,
        output='artifacts/track2a_72_guard_effect_DESCRIPTIVE.json',
    )
    assert paired_guard['case_count'] == 72
    assert paired_guard['new_gpu_inference_performed'] is False
    assert paired_guard['independent_semantic_quality_verified'] is False
    print(json.dumps(paired_guard, indent=2, ensure_ascii=False))
    print('Only paired CPU guard diagnostics saved; no new GPU work or model training.')
else:
    print('Optional paired guard analysis OFF; no CPU replay or output changes.')


## 8 — Owner-only download or recovery snapshot (never publish model weights)

When `EXPORT_RESULTS=True`, this cell re-verifies genuine native AuditKIT saved output before constructing one private ZIP with the frozen challenge, the original model responses, and the three complete evaluation receipts. It is intentionally labeled **NOT INDEPENDENT GPU ATTESTATION / NOT SEMANTIC REVIEW**.

When `EXPORT_PARTIAL_BACKUP=True`, and the final report does **not** yet exist, you can download a **checkpoint-only snapshot**. It is explicitly NOT a 72-case completed model evaluation. **Keep the original Drive-backed checkpoint directory unchanged**: never restore from a partial snapshot over an existing folder, and never edit existing case files.

Use the secure download prompt in Colab and keep the archive private. Do not commit raw licensed weights, credentials or reviewer notes to GitHub.

In [ ]:
from zipfile import ZipFile, ZIP_STORED
from google.colab import files
from scripts.verify_track2a_72_receipts import verify_completed_72
from scripts.challenge72_checkpoint import read_checkpoint
if EXPORT_RESULTS:
    assert original_report.is_file(), "No complete original report to export"
    verify_completed_72(AUDIT_DIR, FROZEN_DATA, RAW_REPORT, root=ROOT)
    output_zip = Path("/content/policyproof_challenge72_original_evidence_NOT_ATTESTED.zip")
    assert not output_zip.exists(), "Refusing to overwrite previous result ZIP"
    manifest = {
        "status": "OWNER_SAVED_ORIGINAL_72_MODEL_OUTPUTS_REPLAYED_NOT_EXTERNALLY_ATTESTED",
        "dataset_sha256": KNOWN_CHALLENGE_SHA256,
        "code_commit": code_sha,
        "original_gpu_provenance_independently_verified": False,
        "human_semantic_review_complete": False,
        "training_performed": False,
        "official_lexsi_submission_completed": False,
    }
    with ZipFile(output_zip, "x", compression=ZIP_STORED) as z:
        z.write(source_file, arcname="frozen_72_challenge.jsonl")
        z.write(original_report, arcname="original_72_raw_predictions.json")
        for name in ("measured_72_report.json", "auditkit_72_receipt.json",
                     "EVIDENCE_COMPLETE.json"):
            z.write(ROOT / AUDIT_DIR / name, arcname="auditkit_72/" + name)
        z.writestr("OWNER_SCOPE_MANIFEST.json",
                   json.dumps(manifest, indent=2, ensure_ascii=False) + "\n")
    # Independently validate the exact freshly saved ZIP before offering it
    # for download. A failed replay preserves the original ZIP for diagnosis;
    # do not relabel it completed or skip this check.
    from scripts.verify_colab_72_bundle import inspect_bundle
    verified_export = inspect_bundle(output_zip)
    assert verified_export["status"] == (
        "VERIFIED_PRIVATE_STAGE12_72_LOCAL_SOURCE_REPLAY_NOT_GPU_ATTESTATION"
    )
    assert verified_export["original_raw_saved_responses_source_replayed"] == 72
    assert verified_export["original_gpu_execution_independently_authenticated"] is False
    assert verified_export["official_lexsi_submission_completed"] is False
    print("Exact ZIP verified before download:", json.dumps(
        verified_export, indent=2, ensure_ascii=False))
    print("Private original-model evidence archive:", output_zip)
    files.download(str(output_zip))
elif EXPORT_PARTIAL_BACKUP:
    assert not original_report.exists(), "Full result exists; use validated complete export"
    from scripts.challenge72_checkpoint import read_checkpoint
    folder, completed, _ = read_checkpoint(
        ROOT, CHECKPOINT, cases, KNOWN_CHALLENGE_SHA256,
        MODEL_SHA, resume=True
    )
    assert 0 < len(completed) < 72, "Only incomplete checkpoint prefixes are backups"
    snapshot = Path("/content/policyproof_challenge72_PARTIAL_UNVERIFIED_checkpoint.zip")
    assert not snapshot.exists(), "Never overwrite existing checkpoint backup"
    with ZipFile(snapshot, "x", compression=ZIP_STORED) as z:
        for checkpoint_file in sorted(folder.iterdir()):
            z.write(checkpoint_file, arcname="challenge72_checkpoint/" + checkpoint_file.name)
        z.writestr("PARTIAL_NOT_EVIDENCE.txt",
                   str(len(completed)) + "/72 saved raw responses, NOT a final result.\n")
    print("PARTIAL original-output checkpoint only:", len(completed), "/72")
    files.download(str(snapshot))
else:
    print("No files exported. Existing private source data remains unchanged.")
print("A private ZIP is NOT a public Hugging Face release or official submission.")

## 9 — Return the evidence and proceed to submission gates

If a genuine 72-case original report and AuditKIT replay are complete, send the **private** `policyproof_challenge72_original_evidence_NOT_ATTESTED.zip` back for independent *local consistency* checking. Keep original runtimes and logs separate in case provenance is questioned. Do not send tokens, private Hugging Face cached weights, or raw reviewer identities.

Separate tasks remain: true native AlignTune 40-step training and saved adapter verification, two qualified independent people reviewing the final 60-case proposed labels, real paired final-60 inference, rights-approved public Hugging Face repositories/collections, and the official Lexsi submission.

**Notebook tests can verify only CPU syntax/guards.** This notebook is not itself a genuine executed GPU experiment.

After downloading the original evidence ZIP, verify it directly from a clean local project checkout (no manual ZIP extraction or new inference):

```bash
python -m pip install "auditkit==1.1.3"
python -m scripts.verify_colab_72_bundle --archive /path/to/policyproof_challenge72_original_evidence_NOT_ATTESTED.zip
```

This checks the six exact source files, original frozen dataset, all saved raw responses, original native AuditKIT score receipts and a new CPU replay. It **does not** import evidence, authenticate the original GPU run, or verify human semantics. Keep the original ZIP and Colab runtime logs unmodified.